# MVTec Defect-Type AUROC Analysis

This notebook computes AUROC for every MVTec `class / defect_type` pair. Each defect-type AUROC compares samples from one anomaly type against `good` samples from the same class and model.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()

TARGET_DATASET = 'mvtec'
RESULT_ROOTS = {
    'adaptclip': ROOT / 'results' / 'adaptclip',
    'anomalyclip': ROOT / 'results' / 'anomalyclip',
    'winclip': ROOT / 'results' / 'winclip',
}
MODEL_ORDER = list(RESULT_ROOTS)

LOW_AUROC_THRESHOLD = 80.0
OUT_DIR = ROOT / 'results' / 'defect_type_analysis' / 'mvtec_all_classes'
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

## 1. Load Score CSVs

In [ ]:
def normalize_path(path):
    return str(path).replace('\\\\', '/').replace('\\', '/')


def defect_type_from_path(path):
    parts = normalize_path(path).split('/')

    if 'test' in parts:
        idx = parts.index('test')
        if idx + 1 < len(parts):
            return parts[idx + 1]

    return 'unknown'


SCORE_COLUMNS = {'dataset', 'sample_id', 'class', 'label', 'image_score', 'query_path'}


def is_score_csv(csv_path):
    try:
        columns = set(pd.read_csv(csv_path, nrows=0).columns)
    except pd.errors.EmptyDataError:
        return False
    return SCORE_COLUMNS.issubset(columns)


def score_csv_paths(root):
    paths = sorted(root.rglob('sample_scores_*.csv'))
    if not paths:
        paths = sorted(root.glob('*.csv'))
    return [path for path in paths if is_score_csv(path)]


def load_scores(model, root):
    frames = []
    for csv_path in score_csv_paths(root):
        df = pd.read_csv(csv_path)
        df['model'] = model
        df['source_file'] = str(csv_path.relative_to(ROOT))
        df['path_norm'] = df['query_path'].map(normalize_path)
        df['defect_type'] = df['query_path'].map(defect_type_from_path)
        frames.append(df)
    if not frames:
        print(f'No score CSVs found for {model}: {root}')
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


scores_all = pd.concat(
    [load_scores(model, root) for model, root in RESULT_ROOTS.items()],
    ignore_index=True,
)

scores_all['label'] = pd.to_numeric(scores_all['label'], errors='coerce')
scores_all['image_score'] = pd.to_numeric(scores_all['image_score'], errors='coerce')
required_non_null = ['dataset', 'sample_id', 'class', 'label', 'image_score', 'query_path']
bad_rows = scores_all[required_non_null].isna().any(axis=1)
if bad_rows.any():
    print(f'Dropping {bad_rows.sum()} malformed score rows with missing required values.')
    display(scores_all.loc[bad_rows, ['source_file', 'dataset', 'sample_id', 'class', 'label', 'image_score', 'query_path']].head(20))

scores_all = scores_all.loc[~bad_rows].copy()
scores_all['dataset'] = scores_all['dataset'].astype(str).str.lower()
scores_all['class'] = scores_all['class'].astype(str)
scores_all['label'] = scores_all['label'].astype(int)
scores_all['image_score'] = scores_all['image_score'].astype(float)
scores_all['sample_id'] = scores_all['sample_id'].astype(str)

mvtec = scores_all[scores_all['dataset'] == TARGET_DATASET].copy()
if mvtec.empty:
    raise ValueError('No MVTec rows found in score CSVs.')

print('rows:', len(mvtec))
print('models:', sorted(mvtec['model'].unique()))
print('classes:', sorted(mvtec['class'].unique()))
mvtec.groupby(['model', 'class', 'defect_type', 'label']).size().rename('n').reset_index()

## 2. Defect-Type AUROC Table

In [ ]:
rows = []

for (model, cls), class_df in mvtec.groupby(['model', 'class'], sort=True):
    good = class_df[class_df['label'] == 0]
    anomalies = class_df[class_df['label'] == 1]
    class_auroc = roc_auc_score(class_df['label'], class_df['image_score']) if class_df['label'].nunique() == 2 else np.nan

    for defect_type, defect_df in anomalies.groupby('defect_type', sort=True):
        eval_df = pd.concat([good, defect_df], ignore_index=True)
        defect_auroc = roc_auc_score(eval_df['label'], eval_df['image_score']) if eval_df['label'].nunique() == 2 else np.nan

        rows.append({
            'model': model,
            'dataset': TARGET_DATASET,
            'class': cls,
            'defect_type': defect_type,
            'class_auroc': class_auroc * 100,
            'defect_auroc': defect_auroc * 100,
            'n_good': len(good),
            'n_defect': len(defect_df),
            'mean_good_score': good['image_score'].mean(),
            'mean_defect_score': defect_df['image_score'].mean(),
            'score_gap': defect_df['image_score'].mean() - good['image_score'].mean(),
            'bottom_anomaly_score': defect_df['image_score'].min(),
            'top_anomaly_score': defect_df['image_score'].max(),
        })

summary = pd.DataFrame(rows)
summary['rank_low_to_high'] = summary.groupby('model')['defect_auroc'].rank(method='dense', ascending=True).astype(int)
summary['low_auroc'] = summary['defect_auroc'] <= LOW_AUROC_THRESHOLD
summary = summary.sort_values(['model', 'defect_auroc', 'class', 'defect_type']).reset_index(drop=True)

summary.to_csv(OUT_DIR / 'mvtec_defect_type_auroc.csv', index=False)
summary

## 3. Low-AUROC Defect Types

In [ ]:
low_table = summary[summary['low_auroc']].copy()
low_table.to_csv(OUT_DIR / 'mvtec_low_defect_type_auroc.csv', index=False)

display_cols = [
    'model', 'class', 'defect_type', 'defect_auroc', 'class_auroc',
    'n_good', 'n_defect', 'mean_good_score', 'mean_defect_score', 'score_gap',
]
low_table[display_cols].sort_values(['model', 'defect_auroc', 'class', 'defect_type'])

## 4. Heatmap by Class and Defect Type

In [ ]:
for model in MODEL_ORDER:
    model_df = summary[summary['model'] == model]
    if model_df.empty:
        continue

    pivot = model_df.pivot_table(index='class', columns='defect_type', values='defect_auroc', aggfunc='mean')
    pivot = pivot.reindex(sorted(pivot.index), axis=0).reindex(sorted(pivot.columns), axis=1)

    fig, ax = plt.subplots(figsize=(max(8, 0.65 * len(pivot.columns)), max(5, 0.35 * len(pivot.index))))
    im = ax.imshow(pivot.to_numpy(), aspect='auto', vmin=0, vmax=100, cmap='viridis')
    ax.set_title(f'{model}: MVTec defect-type AUROC (%)')
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('defect type')
    ax.set_ylabel('class')
    fig.colorbar(im, ax=ax, label='AUROC (%)')
    plt.tight_layout()
    plt.show()

## 5. Sorted Bar Plots

In [ ]:
for model in MODEL_ORDER:
    model_df = summary[summary['model'] == model].sort_values('defect_auroc')
    if model_df.empty:
        continue

    labels = model_df['class'] + ' / ' + model_df['defect_type']
    colors = np.where(model_df['low_auroc'], '#d55e00', '#0072b2')

    fig, ax = plt.subplots(figsize=(10, max(4, 0.24 * len(model_df))))
    ax.barh(labels, model_df['defect_auroc'], color=colors)
    ax.axvline(LOW_AUROC_THRESHOLD, color='black', linestyle='--', linewidth=1)
    ax.set_xlim(0, 100)
    ax.set_xlabel('AUROC (%)')
    ax.set_title(f'{model}: MVTec defect-type AUROC, low to high')
    plt.tight_layout()
    plt.show()

## 6. One Representative Image for Lowest Defect Types

In [ ]:
REPRESENTATIVE_MODEL = 'adaptclip'
TOPK_LOW_DEFECTS = 15

lowest = summary[summary['model'] == REPRESENTATIVE_MODEL].nsmallest(TOPK_LOW_DEFECTS, 'defect_auroc')
representatives = []

for _, row in lowest.iterrows():
    sample_df = mvtec[
        (mvtec['model'] == REPRESENTATIVE_MODEL)
        & (mvtec['class'] == row['class'])
        & (mvtec['defect_type'] == row['defect_type'])
        & (mvtec['label'] == 1)
    ].sort_values('image_score')
    if not sample_df.empty:
        sample = sample_df.iloc[0].copy()
        sample['defect_auroc'] = row['defect_auroc']
        representatives.append(sample)

rep_df = pd.DataFrame(representatives)

if rep_df.empty:
    print('No representative anomaly samples found.')
else:
    ncols = min(5, len(rep_df))
    nrows = int(np.ceil(len(rep_df) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.3 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis('off')

    for ax, (_, row) in zip(axes, rep_df.iterrows()):
        image_path = ROOT / normalize_path(row['query_path'])
        if image_path.exists():
            image = plt.imread(image_path)
            ax.imshow(image, cmap='gray' if image.ndim == 2 else None)
        else:
            ax.text(0.5, 0.5, 'missing image', ha='center', va='center')
        ax.set_title(
            f"{row['class']} / {row['defect_type']}\nAUROC={row['defect_auroc']:.1f}, score={row['image_score']:.4f}",
            fontsize=8,
        )

    fig.suptitle(f'{REPRESENTATIVE_MODEL}: lowest defect-type AUROC examples', y=1.01)
    plt.tight_layout()
    plt.show()

rep_df[['class', 'defect_type', 'defect_auroc', 'sample_id', 'image_score', 'query_path']]

## Exported Files

Saved under `results/defect_type_analysis/mvtec_all_classes/`:

- `mvtec_defect_type_auroc.csv`
- `mvtec_low_defect_type_auroc.csv`